# 第6章 金融时间序列 · 课堂代码

> 本 notebook 与课件《Python金融数据分析 · 第6章 金融时间序列》配套。
> 内容改编自《Python金融大数据分析（第2版）》第8章。

**本章首次使用真实市场数据**：`eod_data.csv`（来源：Refinitiv Eikon，教材配套数据），
包含 2010-01 至 2018-06 共 12 个资产的日频收盘价。

**使用说明**
- 点击单元格，按 `Shift + Enter` 运行；
- 确认本目录下有 `eod_data.csv` 文件。

## 1. 真实数据登场

### 1.1 读入数据

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
df = pd.read_csv("eod_data.csv",    # 与本notebook同目录
                 index_col=0,       # 第0列(日期)作索引
                 parse_dates=True)  # 解析成日期类型
df.shape          # (2216, 12)：8年半，2216个日期

In [ ]:
df.columns.tolist()

In [ ]:
df.head()

### 1.2 数据体检：缺失值在哪里？

不同市场的交易日不完全重合，所以表里有 `NaN`——真实数据的常态。
以 SPY（美股ETF）为准过滤日期：只研究美股交易日。

In [ ]:
df.isna().sum()      # 每列缺失个数

In [ ]:
df = df.dropna(subset=["SPY", "AAPL.O"])  # 只保留美股交易日
df.shape             # (2138, 12)

### 1.3 先看全景：归一化价格

不同资产价格量级差异大（黄金上千、汇率1.4），直接画没法比。
除以各自首日价格再乘100：把"价格"变成"涨跌幅指数"。

In [ ]:
sel = [".SPX", "AAPL.O", "MSFT.O", "AMZN.O", "GLD"]
norm = df[sel] / df[sel].iloc[0] * 100   # 起点=100
norm.plot(figsize=(8, 4), grid=True)

## 2. 收益率：金融数据的"通用语言"

### 2.1 简单收益率 vs 对数收益率

- 简单收益率：`(今日-昨日)/昨日`
- 对数收益率：`ln(今日/昨日)`，可以直接加总，统计建模更友好

In [ ]:
simple = df["SPY"].pct_change()          # (今日-昨日)/昨日
logret = np.log(df["SPY"] / df["SPY"].shift(1))
logret.head()

In [ ]:
rets = np.log(df / df.shift(1))   # 12列一次全算
rets.describe().round(4)          # 每列的统计摘要

### 2.2 年化：从日到年

- 年化均值 ≈ 日均值 × 252
- 年化波动率 = 日波动率 × √252（方差按时间线性累加，标准差按平方根累加）

252 是一年的交易日数，金融计算的常用常数。

In [ ]:
rets.mean() * 252                # 年化平均收益

In [ ]:
rets.std() * np.sqrt(252)        # 年化波动率

### 2.3 重采样：日频 → 月频/年频

价格取 `last`，对数收益取 `sum`——语义不同，别混用。

In [ ]:
df.resample("ME").last()     # 每月最后一个交易日的价格

In [ ]:
rets.resample("ME").sum()    # 对数收益按月加总=月收益

## 3. 滚动统计与波动率聚集

`rolling(n)`：每次取最近 n 个数据算统计量，窗口逐日滑动。
前 n-1 个结果为 `NaN`——数据不足，正常现象。

In [ ]:
spy = rets["SPY"].dropna()
ma21  = spy.rolling(21).mean()   # 21日移动平均
vol21 = spy.rolling(21).std() * np.sqrt(252)  # 滚动年化波动率
vol21.head(25)    # 前20个是NaN：窗口不足

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(spy.index, vol21, label="21日滚动", linewidth=1.2)
ax.plot(spy.index, spy.rolling(252).std() * np.sqrt(252),
        label="252日滚动", linewidth=1.2)
ax.set_title("SPY 滚动年化波动率：波动率聚集清晰可见")
ax.legend()
ax.grid(alpha=0.3)

**波动率聚集**（volatility clustering）：波动率平时低，危机时突然飙升，然后缓慢回落。

实际意义：
- 风险度量（VaR）必须用**最近**的数据，而不是八年平均；
- 21日窗口反应快但噪声大，252日窗口平滑但迟钝——窗口选择是权衡。

## 4. 收益率分布与厚尾

In [ ]:
from scipy.stats import norm

r = rets["SPY"].dropna()
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.hist(r, bins=60, density=True, alpha=0.75, label="SPY日收益率")
xx = np.linspace(r.min(), r.max(), 200)
ax.plot(xx, norm.pdf(xx, r.mean(), r.std()), color="red", label="正态参考")
ax.set_title("SPY 日收益率分布：两端高于红线 = 厚尾")
ax.legend()

In [ ]:
(r.abs() > 0.03).sum()    # |日收益|>3%的天数：27天

按正态分布（日标准差约1%）计算，|r|>3% 几乎不可能发生（理论上不到 1 天），
实际却有 **27 天**——**厚尾**：极端行情远比正态分布预想的频繁。

## 5. 回归分析：beta 是怎么算出来的

CAPM：$r_i = \alpha + \beta \, r_m + \varepsilon$

- β=1 与市场同步；β>1 放大波动；β<1 相对稳健。
- 估计方法：把历史日收益做最小二乘回归（OLS）。

In [ ]:
x = rets[".SPX"].dropna()      # 市场收益
y = rets["AAPL.O"].dropna()    # 苹果收益
beta, alpha = np.polyfit(x, y, 1)   # 一元线性拟合
round(beta, 2)

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 4))
ax.scatter(x, y, s=8, alpha=0.45)
xx = np.linspace(x.min(), x.max(), 50)
ax.plot(xx, alpha + beta * xx, color="red", linewidth=1.5,
        label=f"OLS拟合：beta={beta:.2f}")
ax.set_xlabel(".SPX 日对数收益率")
ax.set_ylabel("AAPL.O 日对数收益率")
ax.set_title("AAPL 对市场（.SPX）的回归")
ax.legend()

**相关系数 vs 回归系数**：
- 相关系数衡量"联动紧密程度"，范围 [-1, 1]；
- beta 衡量"市场涨1%，个股平均涨多少"，可以大于1或为负。

In [ ]:
x.corr(y)    # 相关系数：约0.56

**换个视角：VIX 恐慌指数**——市场恐慌升高时，股价倾向下跌（负相关）。

In [ ]:
v = df[".VIX"].dropna()
s = rets["SPY"].dropna()
v, s = v.align(s, join="inner")
v, s = v.dropna(), s.dropna()
print("相关系数：", round(v.corr(s), 3))

fig, ax = plt.subplots(figsize=(5.2, 4))
ax.scatter(v, s, s=8, alpha=0.45)
ax.set_xlabel("VIX 恐慌指数水平")
ax.set_ylabel("SPY 日对数收益率")
ax.set_title("恐慌指数与市场收益")

## 6. 课后练习

1. 对 MSFT.O 重复 beta 回归，比较与苹果的 beta 大小，
   并结合两家公司的业务特点给出解释。
2. 把滚动波动率窗口从 21 改为 63（一个季度），观察图形平滑程度的变化。
3. 选做：统计 |r|>5% 的天数，查一查这些日期分别对应什么历史事件。

<details>
<summary>练习1 参考答案（点击展开）</summary>

```python
y_ms = rets["MSFT.O"].dropna()
beta_ms, alpha_ms = np.polyfit(x, y_ms, 1)
print("MSFT beta:", round(beta_ms, 2))
print("AAPL beta:", round(beta, 2))
```

</details>

<details>
<summary>练习2 参考答案（点击展开）</summary>

```python
vol63 = spy.rolling(63).std() * np.sqrt(252)
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(spy.index, vol21, label="21日", linewidth=0.9, alpha=0.7)
ax.plot(spy.index, vol63, label="63日", linewidth=1.3)
ax.set_title("滚动年化波动率：窗口越长越平滑")
ax.legend()
ax.grid(alpha=0.3)
```

</details>

<details>
<summary>练习3 参考答案（点击展开）</summary>

```python
extreme = r[r.abs() > 0.05]
print(extreme.sort_values())   # 按跌幅排序，对照财经日历查事件
```

</details>